In [ ]:
!pip install -q \
    pandas \
    numpy \
    scikit-learn \
    plotly \
    gradio \
    pypdf \
    langchain-text-splitters \
    sentence-transformers \
    chromadb \
    transformers \
    accelerate \
    sentencepiece \
    langgraph \
    "mcp[cli]"

print("=" * 60)
print("✅ BioResearch AI packages installed successfully")
print("=" * 60)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.8/393.8 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 57.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 86.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 47.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/6

In [ ]:
import os
import re
import json
import sqlite3
import shutil
import warnings

import numpy as np
import pandas as pd
import plotly.express as px

warnings.filterwarnings("ignore")

print("=" * 60)
print("🧬 BIORESEARCH AI")
print("Multi-Agent Generative AI Research Assistant")
print("=" * 60)

PROJECT_NAME = "BioResearch AI"
DATABASE_NAME = "bioresearch.db"
CHROMA_PATH = "./bioresearch_chroma"
COLLECTION_NAME = "bioresearch_documents"

print(f"Project       : {PROJECT_NAME}")
print(f"Database      : {DATABASE_NAME}")
print(f"Vector Store  : {COLLECTION_NAME}")
print("Status        : Configuration loaded successfully")

🧬 BIORESEARCH AI
Multi-Agent Generative AI Research Assistant
Project       : BioResearch AI
Database      : bioresearch.db
Vector Store  : bioresearch_documents
Status        : Configuration loaded successfully


In [4]:
import os
import sqlite3

# Database configuration
DATABASE_NAME = "bioresearch.db"

# Remove old database if it exists
if os.path.exists(DATABASE_NAME):
    os.remove(DATABASE_NAME)
    print("🗑️ Old database removed.")

# Create new database
conn = sqlite3.connect(DATABASE_NAME)
cursor = conn.cursor()

# ------------------------------------------------------------
# 1. RESEARCHERS TABLE
# ------------------------------------------------------------

cursor.execute("""
CREATE TABLE researchers (
    researcher_id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT NOT NULL,
    department TEXT NOT NULL,
    email TEXT
)
""")

# ------------------------------------------------------------
# 2. EXPERIMENTS TABLE
# ------------------------------------------------------------

cursor.execute("""
CREATE TABLE experiments (
    experiment_id TEXT PRIMARY KEY,
    researcher_id INTEGER,
    experiment_name TEXT NOT NULL,
    sample_id TEXT,
    experiment_date TEXT,
    status TEXT,
    observations TEXT,
    FOREIGN KEY (researcher_id)
        REFERENCES researchers(researcher_id)
)
""")

# ------------------------------------------------------------
# 3. RESULTS TABLE
# ------------------------------------------------------------

cursor.execute("""
CREATE TABLE results (
    result_id INTEGER PRIMARY KEY AUTOINCREMENT,
    experiment_id TEXT,
    parameter TEXT,
    value REAL,
    unit TEXT,
    FOREIGN KEY (experiment_id)
        REFERENCES experiments(experiment_id)
)
""")

# ------------------------------------------------------------
# 4. RESEARCH LOGS TABLE
# ------------------------------------------------------------

cursor.execute("""
CREATE TABLE research_logs (
    log_id INTEGER PRIMARY KEY AUTOINCREMENT,
    experiment_id TEXT,
    action TEXT,
    timestamp TEXT,
    details TEXT,
    FOREIGN KEY (experiment_id)
        REFERENCES experiments(experiment_id)
)
""")

# Save changes
conn.commit()
conn.close()

# ------------------------------------------------------------
# VERIFICATION
# ------------------------------------------------------------

print()
print("=" * 60)
print("🧬 BIORESEARCH AI DATABASE")
print("=" * 60)
print()
print("✅ Database created successfully")
print("📁 Database:", DATABASE_NAME)
print()
print("Tables created:")
print("  ✓ researchers")
print("  ✓ experiments")
print("  ✓ results")
print("  ✓ research_logs")
print()
print("=" * 60)

🗑️ Old database removed.

🧬 BIORESEARCH AI DATABASE

✅ Database created successfully
📁 Database: bioresearch.db

Tables created:
  ✓ researchers
  ✓ experiments
  ✓ results
  ✓ research_logs



In [5]:
conn = sqlite3.connect(DATABASE_NAME)
cursor = conn.cursor()

researchers = [
    (
        "Dr. Priya Sharma",
        "Biotechnology",
        "priya@bioresearch.demo"
    ),
    (
        "Dr. Rahul Mehta",
        "Molecular Biology",
        "rahul@bioresearch.demo"
    ),
    (
        "Dr. Ananya Rao",
        "Bioinformatics",
        "ananya@bioresearch.demo"
    )
]

cursor.executemany("""
INSERT INTO researchers
(name, department, email)
VALUES (?, ?, ?)
""", researchers)

conn.commit()
conn.close()

print("✅ 3 synthetic researchers added")

✅ 3 synthetic researchers added


In [6]:
conn = sqlite3.connect(DATABASE_NAME)
cursor = conn.cursor()

experiments = [
    (
        "EXP001",
        1,
        "Yeast Growth Analysis",
        "SAMPLE001",
        "2026-09-01",
        "Completed",
        "Yeast growth was observed under optimized nutrient conditions."
    ),
    (
        "EXP002",
        2,
        "Enzyme Activity Study",
        "SAMPLE002",
        "2026-09-02",
        "Completed",
        "Enzyme activity increased under moderate temperature conditions."
    ),
    (
        "EXP003",
        1,
        "Protein Stability Test",
        "SAMPLE003",
        "2026-09-03",
        "In Progress",
        "Protein stability was evaluated at different pH conditions."
    ),
    (
        "EXP004",
        1,
        "Cell Growth Observation",
        "SAMPLE004",
        "2026-09-04",
        "Completed",
        "Cell growth was monitored under controlled laboratory conditions."
    ),
    (
        "EXP005",
        3,
        "DNA Sample Observation",
        "SAMPLE005",
        "2026-09-05",
        "Completed",
        "Synthetic DNA sample observations were recorded for demonstration."
    )
]

cursor.executemany("""
INSERT INTO experiments
(
    experiment_id,
    researcher_id,
    experiment_name,
    sample_id,
    experiment_date,
    status,
    observations
)
VALUES (?, ?, ?, ?, ?, ?, ?)
""", experiments)

conn.commit()
conn.close()

print("✅ 5 synthetic experiments added")

✅ 5 synthetic experiments added


In [7]:
conn = sqlite3.connect(DATABASE_NAME)
cursor = conn.cursor()

results = [
    ("EXP001", "Growth Rate", 0.82, "OD/hour"),
    ("EXP001", "Temperature", 30.0, "°C"),
    ("EXP001", "pH", 6.5, "pH"),

    ("EXP002", "Enzyme Activity", 78.5, "U/mL"),
    ("EXP002", "Temperature", 37.0, "°C"),
    ("EXP002", "pH", 7.2, "pH"),

    ("EXP003", "Protein Stability", 91.2, "%"),
    ("EXP003", "Temperature", 25.0, "°C"),
    ("EXP003", "pH", 7.0, "pH"),

    ("EXP004", "Cell Growth", 84.5, "%"),
    ("EXP004", "Temperature", 32.0, "°C"),
    ("EXP004", "pH", 6.8, "pH"),

    ("EXP005", "DNA Quality", 94.3, "%"),
    ("EXP005", "Temperature", 25.0, "°C"),
    ("EXP005", "pH", 7.4, "pH")
]

cursor.executemany("""
INSERT INTO results
(experiment_id, parameter, value, unit)
VALUES (?, ?, ?, ?)
""", results)

conn.commit()
conn.close()

print(f"✅ {len(results)} experimental results added")

✅ 15 experimental results added


In [8]:
conn = sqlite3.connect(DATABASE_NAME)
cursor = conn.cursor()

logs = [
    (
        "EXP001",
        "Experiment Created",
        "2026-09-01 10:00:00",
        "Experiment initialized."
    ),
    (
        "EXP001",
        "Analysis Completed",
        "2026-09-01 15:30:00",
        "Initial growth analysis completed."
    ),
    (
        "EXP002",
        "Experiment Created",
        "2026-09-02 09:30:00",
        "Enzyme activity study initialized."
    ),
    (
        "EXP003",
        "Experiment Started",
        "2026-09-03 11:00:00",
        "Protein stability testing started."
    ),
    (
        "EXP004",
        "Experiment Created",
        "2026-09-04 10:15:00",
        "Cell growth observation started."
    )
]

cursor.executemany("""
INSERT INTO research_logs
(experiment_id, action, timestamp, details)
VALUES (?, ?, ?, ?)
""", logs)

conn.commit()
conn.close()

print(f"✅ {len(logs)} research logs added")

✅ 5 research logs added


In [10]:
import sqlite3
import pandas as pd

DATABASE_NAME = "bioresearch.db"

# Connect to database
conn = sqlite3.connect(DATABASE_NAME)

# Read all tables
researchers_df = pd.read_sql_query(
    "SELECT * FROM researchers",
    conn
)

experiments_df = pd.read_sql_query(
    "SELECT * FROM experiments",
    conn
)

results_df = pd.read_sql_query(
    "SELECT * FROM results",
    conn
)

logs_df = pd.read_sql_query(
    "SELECT * FROM research_logs",
    conn
)

# Close connection
conn.close()

# Display database status
print("=" * 60)
print("🗄️  BIORESEARCH AI - DATABASE STATUS")
print("=" * 60)

print("Researchers :", len(researchers_df))
print("Experiments :", len(experiments_df))
print("Results     :", len(results_df))
print("Logs        :", len(logs_df))

print("=" * 60)

# Display experiment records
print("\n🧪 EXPERIMENT RECORDS")
print("=" * 60)

display(experiments_df)

🗄️  BIORESEARCH AI - DATABASE STATUS
Researchers : 3
Experiments : 5
Results     : 15
Logs        : 5

🧪 EXPERIMENT RECORDS


,experiment_id,researcher_id,experiment_name,sample_id,experiment_date,status,observations
0,EXP001,1,Yeast Growth Analysis,SAMPLE001,2026-09-01,Completed,Yeast growth was observed under optimized nutr...
1,EXP002,2,Enzyme Activity Study,SAMPLE002,2026-09-02,Completed,Enzyme activity increased under moderate tempe...
2,EXP003,1,Protein Stability Test,SAMPLE003,2026-09-03,In Progress,Protein stability was evaluated at different p...
3,EXP004,1,Cell Growth Observation,SAMPLE004,2026-09-04,Completed,Cell growth was monitored under controlled lab...
4,EXP005,3,DNA Sample Observation,SAMPLE005,2026-09-05,Completed,Synthetic DNA sample observations were recorde...


In [11]:
from google.colab import files

print("📚 Upload scientific/research PDFs")
print("You may upload 1 or more PDF files.")
print("If you do not have a PDF, click Cancel.")

uploaded = files.upload()

pdf_files = []

for filename in uploaded.keys():
    if filename.lower().endswith(".pdf"):
        pdf_files.append(filename)

print()
print("📚 PDF FILES AVAILABLE")

if pdf_files:
    for file in pdf_files:
        print("✅", file)
else:
    print("ℹ️ No PDF uploaded.")
    print("Built-in synthetic research knowledge will be used.")

📚 Upload scientific/research PDFs
You may upload 1 or more PDF files.
If you do not have a PDF, click Cancel.


Saving Biotechnology.pdf to Biotechnology.pdf

📚 PDF FILES AVAILABLE
✅ Biotechnology.pdf


In [13]:
# Install pypdf if it is missing
!pip install -q pypdf

# Import PDF reader
from pypdf import PdfReader

# Make sure pdf_files exists
if "pdf_files" not in globals():
    pdf_files = []

documents = {}

print("=" * 60)
print("📄 PDF TEXT EXTRACTION")
print("=" * 60)

# Check whether PDFs were uploaded
if not pdf_files:
    print("ℹ️ No PDF uploaded.")
    print("Using built-in BioResearch AI knowledge in the next cell.")

else:
    for pdf_file in pdf_files:

        print(f"\n📖 Reading: {pdf_file}")

        try:
            reader = PdfReader(pdf_file)

            text = ""

            for page_number, page in enumerate(reader.pages, start=1):

                page_text = page.extract_text()

                if page_text:
                    text += page_text + "\n"

            if text.strip():

                documents[pdf_file] = text

                print(f"✅ Extracted successfully")
                print(f"   Pages : {len(reader.pages)}")
                print(f"   Characters : {len(text)}")

            else:

                print("⚠️ No readable text found in this PDF.")

        except Exception as e:

            print(f"❌ Error reading {pdf_file}")
            print("Error:", e)

print("\n" + "=" * 60)
print("📊 EXTRACTION SUMMARY")
print("=" * 60)

print("PDF files found      :", len(pdf_files))
print("PDFs extracted       :", len(documents))

print("=" * 60)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.8/393.8 kB 5.3 MB/s eta 0:00:00
📄 PDF TEXT EXTRACTION

📖 Reading: Biotechnology.pdf
✅ Extracted successfully
   Pages : 40
   Characters : 88869

📊 EXTRACTION SUMMARY
PDF files found      : 1
PDFs extracted       : 1


In [15]:
# Install required package
!pip install -q langchain-text-splitters

# Import
from langchain_text_splitters import RecursiveCharacterTextSplitter

# ------------------------------------------------------------
# Create text splitter
# ------------------------------------------------------------

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=120,
    separators=["\n\n", "\n", ". ", " ", ""]
)

# ------------------------------------------------------------
# Create document chunks
# ------------------------------------------------------------

chunks = []

# 1️⃣ Add uploaded PDF documents
if "documents" not in globals():
    documents = {}

for source, text in documents.items():

    split_texts = text_splitter.split_text(text)

    for i, chunk_text in enumerate(split_texts):

        if chunk_text.strip():

            chunks.append({
                "text": chunk_text,
                "source": source,
                "chunk_id": i
            })


# ------------------------------------------------------------
# 2️⃣ Built-in biotechnology knowledge
# ------------------------------------------------------------

builtin_knowledge = {

    "Enzyme Activity Knowledge": """
Enzyme activity is influenced by several important factors including
temperature, pH, substrate concentration, enzyme concentration,
inhibitors, cofactors, ionic strength, and reaction time.

Most enzymes have an optimal temperature range where their activity
is highest. Increasing temperature can increase molecular collisions,
but excessive temperature may cause enzyme denaturation.

pH can strongly influence enzyme structure and activity. Different
enzymes have different optimal pH ranges.

Substrate concentration generally increases reaction rate until the
enzyme becomes saturated. At saturation, increasing substrate further
does not produce a proportional increase in reaction rate.

Enzyme inhibitors can reduce enzyme activity. Competitive inhibitors
interact with the active site, while non-competitive inhibitors may
affect enzyme activity through other binding mechanisms.

Cofactors and coenzymes can be required for the biological activity
of some enzymes.
""",

    "Protein Stability Knowledge": """
Protein stability depends on temperature, pH, ionic environment,
solvent conditions, molecular structure, and interactions between
amino acid residues.

High temperatures can disrupt weak interactions that maintain protein
structure and may lead to denaturation.

Extreme pH conditions can alter protein charge and disrupt interactions
within the protein structure.

Protein stability experiments may compare structural stability under
different temperature or pH conditions.

A stable protein generally maintains its functional structure under
the tested experimental conditions.
""",

    "Cell Growth Knowledge": """
Cell growth can be influenced by temperature, pH, nutrient availability,
oxygen concentration, culture conditions, incubation time, and other
environmental factors.

Microbial and cell cultures often have an optimal temperature and pH
range for growth.

Insufficient nutrients can limit cell growth, while unsuitable
environmental conditions can reduce growth rate or cell viability.

Growth measurements can be collected at different time points and
used to analyze growth trends.
""",

    "Biotechnology Experiment Analysis": """
Biotechnology experiments generate structured measurements such as
temperature, pH, growth rate, enzyme activity, protein stability,
DNA quality, and other experimental parameters.

Statistical analysis can be used to calculate mean, minimum, maximum,
standard deviation, and other descriptive measures.

Machine learning can also be demonstrated using synthetic experimental
data to identify relationships between input variables and predicted
experimental outcomes.

Experimental conclusions should be based on available measurements
and should not invent observations that were not recorded.
"""
}

# ------------------------------------------------------------
# Convert built-in knowledge into chunks
# ------------------------------------------------------------

builtin_chunk_id = 0

for source, text in builtin_knowledge.items():

    split_texts = text_splitter.split_text(text)

    for chunk_text in split_texts:

        if chunk_text.strip():

            chunks.append({
                "text": chunk_text,
                "source": source,
                "chunk_id": builtin_chunk_id
            })

            builtin_chunk_id += 1


# ------------------------------------------------------------
# Display result
# ------------------------------------------------------------

print("=" * 60)
print("📚 BIORESEARCH AI — DOCUMENT CHUNKING")
print("=" * 60)

print("Uploaded PDFs       :", len(documents))
print("Total chunks        :", len(chunks))

print("=" * 60)

# Show first few chunks
for i, chunk in enumerate(chunks[:5]):

    print(f"\n🔹 CHUNK {i + 1}")
    print("Source:", chunk["source"])
    print("Chunk ID:", chunk["chunk_id"])
    print("Text:", chunk["text"][:300], "...")

print("\n" + "=" * 60)
print("✅ DOCUMENT CHUNKING COMPLETED")
print("=" * 60)

📚 BIORESEARCH AI — DOCUMENT CHUNKING
Uploaded PDFs       : 1
Total chunks        : 143

🔹 CHUNK 1
Source: Biotechnology.pdf
Chunk ID: 0
Text: BIOTECHNOLOGY
 
HIMACHAL	PRADESH	 in Himachal Pradesh
SHIMLA­171002 
Status Report 
2017 
	
	
	 	 	 	 	 	 	 	 	 	 	
 
 
 
 
 
                 
                   
             
                 
             
                 
                 
                
     
              
                 ...

🔹 CHUNK 2
Source: Biotechnology.pdf
Chunk ID: 1
Text: Food	Civil	Supplies	&	Consumer	Aﬀairs
Government	of	Himachal	Pradesh 
Foreword 
The	 biotechnology	 sector	 of	 India	 is	 highly	 innovative	 and	 growing.	 India	 is	 among	 the	 top	 12	 biotech	 
destinations	 in	 the	 world	 and	 ranks	 third	 in	 the	 Asia­Pacific	 region.	 Out	 of	 the	 top	  ...

🔹 CHUNK 3
Source: Biotechnology.pdf
Chunk ID: 2
Text: bio­agri	 (14	 per	 cent),	 bio­industry	 (3	 per	 cent),	 and	 bioinformatics	 contributing	 (1	 per	 cent).	
India	 has	 no	 dearth	 of	

In [16]:
from sentence_transformers import SentenceTransformer

print("🧠 Loading transformer embedding model...")

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

chunk_texts = [
    chunk["text"]
    for chunk in chunks
]

if chunk_texts:

    embeddings = embedding_model.encode(
        chunk_texts,
        show_progress_bar=True,
        convert_to_numpy=True
    )

    print()
    print("✅ Embeddings created")
    print("Number of vectors :", len(embeddings))
    print("Vector dimension   :", embeddings.shape[1])

else:

    embeddings = np.empty((0, 384))

    print("⚠️ No document chunks available.")

🧠 Loading transformer embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/5 [00:00<?, ?it/s]


✅ Embeddings created
Number of vectors : 143
Vector dimension   : 384


In [18]:
# Install ChromaDB
!pip install -q chromadb

# Import required libraries
import os
import shutil
import chromadb

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

CHROMA_PATH = "./bioresearch_chroma"
COLLECTION_NAME = "bioresearch_documents"

# ------------------------------------------------------------
# Remove old ChromaDB from previous attempts
# ------------------------------------------------------------

if os.path.exists(CHROMA_PATH):
    print("🧹 Removing old ChromaDB...")
    shutil.rmtree(CHROMA_PATH)

# ------------------------------------------------------------
# Create persistent ChromaDB client
# ------------------------------------------------------------

client = chromadb.PersistentClient(
    path=CHROMA_PATH
)

# ------------------------------------------------------------
# Create collection
# ------------------------------------------------------------

collection = client.get_or_create_collection(
    name=COLLECTION_NAME
)

# ------------------------------------------------------------
# Display status
# ------------------------------------------------------------

print("=" * 60)
print("🧠 BIORESEARCH AI — CHROMADB")
print("=" * 60)

print("Database path :", CHROMA_PATH)
print("Collection    :", COLLECTION_NAME)
print("Documents     :", collection.count())

print("=" * 60)
print("✅ ChromaDB created successfully!")
print("=" * 60)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 3.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently ta

In [19]:
if len(chunks) > 0:

    ids = [
        f"research_chunk_{i}"
        for i in range(len(chunks))
    ]

    metadatas = [
        {
            "source": chunk["source"],
            "chunk_id": int(chunk["chunk_id"])
        }
        for chunk in chunks
    ]

    collection.upsert(
        ids=ids,
        documents=chunk_texts,
        embeddings=embeddings.tolist(),
        metadatas=metadatas
    )

    print("✅ Documents stored in ChromaDB")
    print("Vector count:", collection.count())

else:

    print("⚠️ No chunks available.")

✅ Documents stored in ChromaDB
Vector count: 143


In [20]:
def search_research_documents(
    query,
    top_k=3
):
    """
    Retrieve the most relevant research documents
    using semantic similarity.
    """

    if not query or not query.strip():
        return []

    if collection.count() == 0:
        return []

    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True
    )[0]

    results = collection.query(
        query_embeddings=[
            query_embedding.tolist()
        ],
        n_results=min(
            top_k,
            collection.count()
        )
    )

    retrieved_documents = []

    if results.get("documents"):

        documents_result = results["documents"][0]

        metadatas_result = (
            results.get("metadatas", [[]])[0]
        )

        distances_result = (
            results.get("distances", [[]])[0]
        )

        for i, document in enumerate(documents_result):

            metadata = (
                metadatas_result[i]
                if i < len(metadatas_result)
                else {}
            )

            distance = (
                distances_result[i]
                if i < len(distances_result)
                else None
            )

            retrieved_documents.append({
                "text": document,
                "source": metadata.get(
                    "source",
                    "Unknown"
                ),
                "chunk_id": metadata.get(
                    "chunk_id",
                    -1
                ),
                "distance": distance
            })

    return retrieved_documents


print("=" * 60)
print("🔎 RAG FUNCTION READY")
print("=" * 60)

print(
    "Function exists:",
    "search_research_documents" in globals()
)

print(
    "ChromaDB documents:",
    collection.count()
)

🔎 RAG FUNCTION READY
Function exists: True
ChromaDB documents: 143


In [21]:
from transformers import pipeline

print("🤖 Loading BioResearch AI language model...")
print("Please wait during the first download.")

generator = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-0.5B-Instruct",
    max_new_tokens=60,
    do_sample=False
)

print()
print("✅ LLM loaded successfully")
print("Model: Qwen/Qwen2.5-0.5B-Instruct")

🤖 Loading BioResearch AI language model...
Please wait during the first download.


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

...

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.



✅ LLM loaded successfully
Model: Qwen/Qwen2.5-0.5B-Instruct


In [22]:
def get_experiment(experiment_id):

    conn = sqlite3.connect(DATABASE_NAME)

    query = """
    SELECT
        e.experiment_id,
        r.name AS researcher,
        r.department,
        e.experiment_name,
        e.sample_id,
        e.experiment_date,
        e.status,
        e.observations
    FROM experiments e
    LEFT JOIN researchers r
        ON e.researcher_id = r.researcher_id
    WHERE e.experiment_id = ?
    """

    experiment = pd.read_sql_query(
        query,
        conn,
        params=(experiment_id.upper(),)
    )

    conn.close()

    return experiment


def get_experiment_results(experiment_id):

    conn = sqlite3.connect(DATABASE_NAME)

    query = """
    SELECT
        parameter,
        value,
        unit
    FROM results
    WHERE experiment_id = ?
    """

    results = pd.read_sql_query(
        query,
        conn,
        params=(experiment_id.upper(),)
    )

    conn.close()

    return results


def experiment_agent(experiment_id):

    experiment_id = experiment_id.upper().strip()

    experiment = get_experiment(experiment_id)

    if experiment.empty:

        return (
            f"❌ Experiment `{experiment_id}` "
            f"was not found."
        )

    results = get_experiment_results(
        experiment_id
    )

    exp = experiment.iloc[0]

    response = f"""
🧪 EXPERIMENT DETAILS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Experiment ID : {exp['experiment_id']}
Researcher    : {exp['researcher']}
Department    : {exp['department']}

Experiment    : {exp['experiment_name']}
Sample ID     : {exp['sample_id']}
Date          : {exp['experiment_date']}
Status        : {exp['status']}

📝 Observations
{exp['observations']}

📊 Results
"""

    if results.empty:

        response += "\nNo experimental results available."

    else:

        for _, row in results.iterrows():

            response += (
                f"\n• {row['parameter']}: "
                f"{row['value']} {row['unit']}"
            )

    return response


print("✅ Experiment Agent ready")

print()
print(experiment_agent("EXP001"))

✅ Experiment Agent ready


🧪 EXPERIMENT DETAILS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Experiment ID : EXP001
Researcher    : Dr. Priya Sharma
Department    : Biotechnology

Experiment    : Yeast Growth Analysis
Sample ID     : SAMPLE001
Date          : 2026-09-01
Status        : Completed

📝 Observations
Yeast growth was observed under optimized nutrient conditions.

📊 Results

• Growth Rate: 0.82 OD/hour
• Temperature: 30.0 °C
• pH: 6.5 pH


In [23]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score


def analyze_experiment(experiment_id):

    experiment_id = experiment_id.upper().strip()

    conn = sqlite3.connect(DATABASE_NAME)

    query = """
    SELECT
        parameter,
        value,
        unit
    FROM results
    WHERE experiment_id = ?
    """

    df = pd.read_sql_query(
        query,
        conn,
        params=(experiment_id,)
    )

    conn.close()

    if df.empty:

        return {
            "status": "error",
            "message": (
                f"No results found for "
                f"{experiment_id}"
            )
        }

    values = df["value"].astype(float)

    statistics = {
        "count": int(len(values)),
        "mean": float(values.mean()),
        "minimum": float(values.min()),
        "maximum": float(values.max()),
        "standard_deviation": float(
            values.std()
            if len(values) > 1
            else 0
        )
    }

    return {
        "status": "success",
        "experiment_id": experiment_id,
        "data": df,
        "statistics": statistics
    }


# ------------------------------------------------------------
# ML DEMONSTRATION
# ------------------------------------------------------------

def run_ml_prediction():

    # Synthetic training data
    training_data = pd.DataFrame({
        "temperature": [
            20, 22, 25, 27, 30,
            32, 35, 37, 40, 42,
            45, 48
        ],
        "pH": [
            6.0, 6.2, 6.5, 6.7,
            6.9, 7.0, 7.1, 7.2,
            7.3, 7.4, 7.5, 7.6
        ],
        "substrate": [
            20, 25, 30, 35,
            40, 45, 50, 55,
            60, 65, 70, 75
        ],
        "activity": [
            42, 48, 57, 64,
            71, 76, 81, 84,
            82, 77, 69, 58
        ]
    })

    X = training_data[
        [
            "temperature",
            "pH",
            "substrate"
        ]
    ]

    y = training_data["activity"]

    model = RandomForestRegressor(
        n_estimators=100,
        random_state=42
    )

    model.fit(X, y)

    # Demonstration prediction
    test_sample = pd.DataFrame({
        "temperature": [37],
        "pH": [7.2],
        "substrate": [55]
    })

    prediction = model.predict(
        test_sample
    )[0]

    training_prediction = model.predict(X)

    mae = mean_absolute_error(
        y,
        training_prediction
    )

    r2 = r2_score(
        y,
        training_prediction
    )

    return {
        "predicted_activity": float(prediction),
        "mae": float(mae),
        "r2_score": float(r2),
        "model": "Random Forest Regressor"
    }


def analysis_agent(experiment_id):

    analysis = analyze_experiment(
        experiment_id
    )

    if analysis["status"] == "error":
        return analysis["message"]

    stats = analysis["statistics"]

    ml_result = run_ml_prediction()

    return f"""
📊 ANALYSIS AGENT
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Experiment ID : {experiment_id.upper()}

Statistical Analysis
--------------------
Number of Results : {stats['count']}
Mean             : {stats['mean']:.2f}
Minimum          : {stats['minimum']:.2f}
Maximum          : {stats['maximum']:.2f}
Std. Deviation   : {stats['standard_deviation']:.2f}

🤖 Machine Learning
--------------------
Model : {ml_result['model']}

Predicted Enzyme Activity
: {ml_result['predicted_activity']:.2f} U/mL

Model MAE
: {ml_result['mae']:.2f}

Model R²
: {ml_result['r2_score']:.2f}

⚠️ ML values are based on synthetic
demonstration data and are not laboratory
predictions.
"""


print("✅ Analysis + ML Agent ready")

print(analysis_agent("EXP002"))

✅ Analysis + ML Agent ready

📊 ANALYSIS AGENT
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Experiment ID : EXP002

Statistical Analysis
--------------------
Number of Results : 3
Mean             : 40.90
Minimum          : 7.20
Maximum          : 78.50
Std. Deviation   : 35.81

🤖 Machine Learning
--------------------
Model : Random Forest Regressor

Predicted Enzyme Activity
: 82.90 U/mL

Model MAE
: 1.32

Model R²
: 0.98

⚠️ ML values are based on synthetic
demonstration data and are not laboratory
predictions.



In [24]:
def rag_agent(query, top_k=3):

    retrieved_documents = (
        search_research_documents(
            query,
            top_k
        )
    )

    if not retrieved_documents:

        return {
            "answer": (
                "I could not find relevant research "
                "information."
            ),
            "sources": []
        }

    context_parts = []

    sources = []

    for doc in retrieved_documents:

        context_parts.append(
            doc["text"]
        )

        source = doc["source"]

        if source not in sources:
            sources.append(source)

    context = "\n\n".join(
        context_parts
    )

    prompt = f"""
You are BioResearch AI, a biotechnology
research assistant.

Answer the researcher's question using
the provided research context.

Question:
{query}

Research Context:
{context}

Give a concise scientific explanation.
Do not invent experimental facts.
"""

    try:

        result = generator(
            prompt
        )

        generated_text = result[0]["generated_text"]

        if generated_text.startswith(prompt):
            answer = generated_text[
                len(prompt):
            ].strip()
        else:
            answer = generated_text.strip()

    except Exception:

        # Reliable fallback
        answer = context[:1200]

    return {
        "answer": answer,
        "sources": sources,
        "documents": retrieved_documents
    }


print("✅ RAG Agent ready")

rag_test = rag_agent(
    "What factors affect enzyme activity?"
)

print("\nAnswer:")
print(rag_test["answer"])

print("\nSources:")
for source in rag_test["sources"]:
    print("•", source)

[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Both `max_new_tokens` (=60) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


✅ RAG Agent ready


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.



Answer:
Use words from the Research Context.
Avoid repetition of information already present in the text.
Ensure clarity and conciseness in your response.
Summarize the key points in bullet points format.
Provide examples if possible.
Include at least one example of how each factor affects enzyme activity.
Discuss any limitations of

Sources:
• Enzyme Activity Knowledge
• Protein Stability Knowledge


In [25]:
def generate_research_report(
    experiment_id
):

    experiment = get_experiment(
        experiment_id
    )

    if experiment.empty:

        return (
            f"❌ Experiment "
            f"{experiment_id.upper()} "
            f"does not exist."
        )

    results = get_experiment_results(
        experiment_id
    )

    analysis = analyze_experiment(
        experiment_id
    )

    exp = experiment.iloc[0]

    report = f"""
╔════════════════════════════════════════════╗
       🧬 BIORESEARCH AI REPORT
╚════════════════════════════════════════════╝

Experiment ID
{exp['experiment_id']}

Experiment Name
{exp['experiment_name']}

Researcher
{exp['researcher']}

Department
{exp['department']}

Sample
{exp['sample_id']}

Date
{exp['experiment_date']}

Status
{exp['status']}

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
OBSERVATIONS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

{exp['observations']}

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
EXPERIMENTAL RESULTS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
"""

    for _, row in results.iterrows():

        report += (
            f"\n• {row['parameter']}: "
            f"{row['value']} {row['unit']}"
        )

    if analysis["status"] == "success":

        stats = analysis["statistics"]

        report += f"""

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
STATISTICAL SUMMARY
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Mean           : {stats['mean']:.2f}
Minimum        : {stats['minimum']:.2f}
Maximum        : {stats['maximum']:.2f}
Std. Deviation : {stats['standard_deviation']:.2f}

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
SYSTEM NOTE
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

This report was generated using the
BioResearch AI multi-agent research system.

Data used in this demonstration is synthetic.
"""

    return report


print("✅ Report Agent ready")

print(
    generate_research_report("EXP002")
)

✅ Report Agent ready

╔════════════════════════════════════════════╗
       🧬 BIORESEARCH AI REPORT
╚════════════════════════════════════════════╝

Experiment ID
EXP002

Experiment Name
Enzyme Activity Study

Researcher
Dr. Rahul Mehta

Department
Molecular Biology

Sample
SAMPLE002

Date
2026-09-02

Status
Completed

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
OBSERVATIONS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Enzyme activity increased under moderate temperature conditions.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
EXPERIMENTAL RESULTS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

• Enzyme Activity: 78.5 U/mL
• Temperature: 37.0 °C
• pH: 7.2 pH

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
STATISTICAL SUMMARY
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Mean           : 40.90
Minimum        : 7.20
Maximum        : 78.50
Std. Deviation : 35.81

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
SYSTEM NOTE
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

This report was generated 

In [26]:
try:

    from mcp.server import MCPServer

    mcp = MCPServer(
        "BioResearch AI MCP Server"
    )

    @mcp.tool()
    def search_research_documents_mcp(
        query: str,
        top_k: int = 3
    ) -> list:

        return search_research_documents(
            query,
            top_k
        )


    @mcp.tool()
    def get_experiment_mcp(
        experiment_id: str
    ) -> str:

        return experiment_agent(
            experiment_id
        )


    @mcp.tool()
    def analyze_experiment_mcp(
        experiment_id: str
    ) -> str:

        return analysis_agent(
            experiment_id
        )


    @mcp.tool()
    def generate_report_mcp(
        experiment_id: str
    ) -> str:

        return generate_research_report(
            experiment_id
        )


    print("=" * 60)
    print("🔧 MCP TOOL LAYER READY")
    print("=" * 60)
    print("✓ search_research_documents_mcp")
    print("✓ get_experiment_mcp")
    print("✓ analyze_experiment_mcp")
    print("✓ generate_report_mcp")

except Exception as e:

    print("⚠️ MCP import issue:", e)
    print(
        "The remaining BioResearch AI system "
        "can still run."
    )

⚠️ MCP import issue: No module named 'mcp'
The remaining BioResearch AI system can still run.


In [27]:
from typing import TypedDict, List, Dict, Any

from langgraph.graph import (
    StateGraph,
    START,
    END
)


class ResearchState(TypedDict, total=False):

    user_query: str
    experiment_id: str

    retrieved_documents: List[
        Dict[str, Any]
    ]

    experiment_data: Dict[str, Any]

    analysis: Dict[str, Any]

    final_report: str

    next_agent: str


# ------------------------------------------------------------
# SUPERVISOR
# ------------------------------------------------------------

def supervisor_agent(state: ResearchState):

    query = state.get(
        "user_query",
        ""
    ).strip()

    query_lower = query.lower()

    # Analysis intent
    analysis_words = [
        "analyze",
        "analysis",
        "statistic",
        "statistics",
        "mean",
        "minimum",
        "maximum",
        "machine learning",
        "predict",
        "prediction",
        "ml"
    ]

    if any(
        word in query_lower
        for word in analysis_words
    ):

        return {
            "next_agent": "analysis"
        }


    # Experiment ID intent
    match = re.search(
        r"\bEXP\d{3}\b",
        query.upper()
    )

    if match:

        return {
            "experiment_id": match.group(0),
            "next_agent": "experiment"
        }


    # Research / knowledge intent
    return {
        "next_agent": "rag"
    }


# ------------------------------------------------------------
# RAG NODE
# ------------------------------------------------------------

def rag_node(state: ResearchState):

    query = state.get(
        "user_query",
        ""
    )

    result = rag_agent(
        query,
        top_k=3
    )

    answer = result["answer"]

    sources = result["sources"]

    if sources:

        answer += (
            "\n\n📚 Sources:\n"
            + "\n".join(
                f"• {source}"
                for source in sources
            )
        )

    return {
        "retrieved_documents":
            result.get(
                "documents",
                []
            ),
        "final_report": answer,
        "next_agent": "end"
    }


# ------------------------------------------------------------
# EXPERIMENT NODE
# ------------------------------------------------------------

def experiment_node(
    state: ResearchState
):

    experiment_id = state.get(
        "experiment_id",
        ""
    )

    response = experiment_agent(
        experiment_id
    )

    return {
        "final_report": response,
        "next_agent": "end"
    }


# ------------------------------------------------------------
# ANALYSIS NODE
# ------------------------------------------------------------

def analysis_node(
    state: ResearchState
):

    experiment_id = state.get(
        "experiment_id",
        ""
    )

    response = analysis_agent(
        experiment_id
    )

    return {
        "final_report": response,
        "next_agent": "end"
    }


# ------------------------------------------------------------
# ROUTER
# ------------------------------------------------------------

def route_supervisor(
    state: ResearchState
):

    return state.get(
        "next_agent",
        "rag"
    )


# ------------------------------------------------------------
# BUILD LANGGRAPH
# ------------------------------------------------------------

workflow = StateGraph(
    ResearchState
)

workflow.add_node(
    "supervisor",
    supervisor_agent
)

workflow.add_node(
    "rag",
    rag_node
)

workflow.add_node(
    "experiment",
    experiment_node
)

workflow.add_node(
    "analysis",
    analysis_node
)

workflow.add_edge(
    START,
    "supervisor"
)

workflow.add_conditional_edges(
    "supervisor",
    route_supervisor,
    {
        "rag": "rag",
        "experiment": "experiment",
        "analysis": "analysis"
    }
)

workflow.add_edge(
    "rag",
    END
)

workflow.add_edge(
    "experiment",
    END
)

workflow.add_edge(
    "analysis",
    END
)

app = workflow.compile()

print("=" * 60)
print("🕸️ LANGGRAPH SUPERVISOR READY")
print("=" * 60)

print("Agents:")
print("✓ Supervisor Agent")
print("✓ RAG Agent")
print("✓ Experiment Agent")
print("✓ Analysis + ML Agent")

print()
print("✓ LangGraph workflow compiled successfully")

🕸️ LANGGRAPH SUPERVISOR READY
Agents:
✓ Supervisor Agent
✓ RAG Agent
✓ Experiment Agent
✓ Analysis + ML Agent

✓ LangGraph workflow compiled successfully


In [29]:
import re

# ------------------------------------------------------------
# Extract Experiment ID
# ------------------------------------------------------------

def extract_experiment_id(query):

    if not query:
        return None

    match = re.search(
        r"\bEXP\d{3}\b",
        query.upper()
    )

    if match:
        return match.group(0)

    return None


# ------------------------------------------------------------
# Detect which agent should handle the query
# ------------------------------------------------------------

def detect_agent(query):

    query_lower = query.lower()

    # Report requests
    report_words = [
        "report",
        "generate report",
        "research report",
        "summary report"
    ]

    if any(word in query_lower for word in report_words):
        return "report"

    # Analysis requests
    analysis_words = [
        "analyze",
        "analysis",
        "statistics",
        "statistical",
        "mean",
        "minimum",
        "maximum",
        "predict",
        "prediction",
        "machine learning",
        "ml"
    ]

    if any(word in query_lower for word in analysis_words):
        return "analysis"

    # Experiment requests
    if extract_experiment_id(query):
        return "experiment"

    # Everything else goes to RAG
    return "rag"


# ------------------------------------------------------------
# Main BioResearch AI Engine
# ------------------------------------------------------------

def bioresearch_ai(user_query):

    if not user_query or not user_query.strip():

        return "⚠️ Please enter a research question."


    # Detect experiment ID
    experiment_id = extract_experiment_id(user_query)

    # Detect agent
    agent = detect_agent(user_query)

    # --------------------------------------------------------
    # EXPERIMENT AGENT
    # --------------------------------------------------------

    if agent == "experiment":

        try:

            response = experiment_agent(
                experiment_id
            )

            return (
                "🤖 **Supervisor Decision: EXPERIMENT**\n\n"
                + response
            )

        except Exception as e:

            return (
                "❌ Experiment Agent Error:\n"
                + str(e)
            )


    # --------------------------------------------------------
    # ANALYSIS AGENT
    # --------------------------------------------------------

    elif agent == "analysis":

        try:

            if experiment_id:

                response = analysis_agent(
                    experiment_id
                )

            else:

                response = analysis_agent(
                    None
                )

            return (
                "🤖 **Supervisor Decision: ANALYSIS**\n\n"
                + response
            )

        except Exception as e:

            return (
                "❌ Analysis Agent Error:\n"
                + str(e)
            )


    # --------------------------------------------------------
    # REPORT AGENT
    # --------------------------------------------------------

    elif agent == "report":

        try:

            if not experiment_id:

                return (
                    "🤖 **Supervisor Decision: REPORT**\n\n"
                    "⚠️ Please specify an experiment ID.\n\n"
                    "Example:\n"
                    "`Generate a report for EXP002`"
                )

            response = generate_research_report(
                experiment_id
            )

            return (
                "🤖 **Supervisor Decision: REPORT**\n\n"
                + response
            )

        except Exception as e:

            return (
                "❌ Report Agent Error:\n"
                + str(e)
            )


    # --------------------------------------------------------
    # RAG RESEARCH AGENT
    # --------------------------------------------------------

    elif agent == "rag":

        try:

            result = rag_agent(
                user_query
            )

            if isinstance(result, dict):

                answer = result.get(
                    "answer",
                    "No answer generated."
                )

                sources = result.get(
                    "sources",
                    []
                )

                output = (
                    "🤖 **Supervisor Decision: RAG RESEARCH**\n\n"
                    + answer
                )

                if sources:

                    output += (
                        "\n\n📚 **Sources:**\n"
                    )

                    for source in sources:

                        output += (
                            f"- {source}\n"
                        )

                return output

            else:

                return (
                    "🤖 **Supervisor Decision: RAG RESEARCH**\n\n"
                    + str(result)
                )

        except Exception as e:

            return (
                "❌ RAG Agent Error:\n"
                + str(e)
            )


# ============================================================
# TEST BIORESEARCH AI
# ============================================================

print("=" * 60)
print("🧬 BIORESEARCH AI ENGINE READY")
print("=" * 60)


print("\nTEST 1")
print("-" * 60)

print(
    bioresearch_ai(
        "Show me EXP001"
    )
)


print("\nTEST 2")
print("-" * 60)

print(
    bioresearch_ai(
        "Analyze EXP002"
    )
)


print("\nTEST 3")
print("-" * 60)

print(
    bioresearch_ai(
        "What factors affect enzyme activity?"
    )
)


print("\nTEST 4")
print("-" * 60)

print(
    bioresearch_ai(
        "Generate a report for EXP002"
    )
)


print("\n" + "=" * 60)
print("✅ BIORESEARCH AI ENGINE TEST COMPLETED")
print("=" * 60)

🧬 BIORESEARCH AI ENGINE READY

TEST 1
------------------------------------------------------------
🤖 **Supervisor Decision: EXPERIMENT**


🧪 EXPERIMENT DETAILS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Experiment ID : EXP001
Researcher    : Dr. Priya Sharma
Department    : Biotechnology

Experiment    : Yeast Growth Analysis
Sample ID     : SAMPLE001
Date          : 2026-09-01
Status        : Completed

📝 Observations
Yeast growth was observed under optimized nutrient conditions.

📊 Results

• Growth Rate: 0.82 OD/hour
• Temperature: 30.0 °C
• pH: 6.5 pH

TEST 2
------------------------------------------------------------


[transformers] Both `max_new_tokens` (=60) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🤖 **Supervisor Decision: ANALYSIS**


📊 ANALYSIS AGENT
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Experiment ID : EXP002

Statistical Analysis
--------------------
Number of Results : 3
Mean             : 40.90
Minimum          : 7.20
Maximum          : 78.50
Std. Deviation   : 35.81

🤖 Machine Learning
--------------------
Model : Random Forest Regressor

Predicted Enzyme Activity
: 82.90 U/mL

Model MAE
: 1.32

Model R²
: 0.98

⚠️ ML values are based on synthetic
demonstration data and are not laboratory
predictions.


TEST 3
------------------------------------------------------------
🤖 **Supervisor Decision: RAG RESEARCH**

Use words from the Research Context.
Avoid repetition of information already present in the text.
Ensure clarity and conciseness in your response.
Summarize the key points in bullet points format.
Provide examples if possible.
Include at least one example of how each factor affects enzyme activity.
Discuss any limitations of

📚 **Sources:**
- Enzyme Activity Knowledge


In [30]:
import gradio as gr


# ------------------------------------------------------------
# CHAT FUNCTION
# ------------------------------------------------------------

def chat_with_ai(
    message,
    history
):

    if not message or not message.strip():

        return history, ""

    try:

        response = bioresearch_ai(
            message
        )

    except Exception as e:

        response = (
            f"❌ Error:\n{str(e)}"
        )

    history = history or []

    history.append({
        "role": "user",
        "content": message
    })

    history.append({
        "role": "assistant",
        "content": response
    })

    return history, ""


# ------------------------------------------------------------
# QUICK ACTION
# ------------------------------------------------------------

def quick_question(question):

    return bioresearch_ai(
        question
    )


# ------------------------------------------------------------
# EXPERIMENT DASHBOARD
# ------------------------------------------------------------

def experiment_dashboard():

    conn = sqlite3.connect(
        DATABASE_NAME
    )

    df = pd.read_sql_query(
        """
        SELECT
            experiment_id AS ID,
            experiment_name AS Experiment,
            sample_id AS Sample,
            experiment_date AS Date,
            status AS Status
        FROM experiments
        ORDER BY experiment_date
        """,
        conn
    )

    conn.close()

    return df


# ------------------------------------------------------------
# CUSTOM CSS
# ------------------------------------------------------------

custom_css = """

body {
    background: #f4f7fb;
}

.gradio-container {
    max-width: 1250px !important;
}

.hero {
    padding: 25px;
    border-radius: 22px;
    background: linear-gradient(
        135deg,
        #172554,
        #312e81,
        #581c87
    );
    color: white;
    text-align: center;
    margin-bottom: 18px;
}

.hero h1 {
    font-size: 38px;
    margin-bottom: 5px;
}

.hero p {
    font-size: 17px;
    opacity: 0.9;
}

.card {
    border-radius: 18px;
    padding: 18px;
    background: white;
    border: 1px solid #e5e7eb;
    min-height: 100px;
}

.section-title {
    font-size: 22px;
    font-weight: 700;
    margin-top: 12px;
}

"""

# ------------------------------------------------------------
# BUILD UI
# ------------------------------------------------------------

with gr.Blocks(
    title="BioResearch AI"
) as demo:

    # HERO
    gr.HTML("""
    <div class="hero">
        <h1>🧬 BioResearch AI</h1>
        <p>
        Multi-Agent Generative AI Assistant
        for Biotechnology Research
        </p>
        <p>
        AI • ML • DL • GenAI • LLM • RAG •
        LangGraph • MCP • Multi-Agent
        </p>
    </div>
    """)


    # TECHNOLOGY CARDS
    with gr.Row():

        gr.HTML("""
        <div class="card">
            <h3>🤖 GenAI</h3>
            <p>Natural language research assistance</p>
        </div>
        """)

        gr.HTML("""
        <div class="card">
            <h3>📚 RAG</h3>
            <p>Research document retrieval</p>
        </div>
        """)

        gr.HTML("""
        <div class="card">
            <h3>🕸️ LangGraph</h3>
            <p>Supervisor workflow</p>
        </div>
        """)

        gr.HTML("""
        <div class="card">
            <h3>🔧 MCP</h3>
            <p>Research tool layer</p>
        </div>
        """)


    # CHAT SECTION
    gr.Markdown(
        "## 💬 Research Assistant"
    )

    chatbot = gr.Chatbot(
        label="🧬 BioResearch AI Assistant",
        height=500
    )

    with gr.Row():

        message_box = gr.Textbox(
            label="Research Query",
            placeholder=(
                "Example: Show me EXP001 "
                "or What factors affect enzyme activity?"
            ),
            lines=2,
            scale=8
        )

        ask_button = gr.Button(
            "🔍 Ask AI",
            variant="primary",
            scale=2
        )


    clear_button = gr.Button(
        "🗑️ Clear Conversation"
    )


    # QUICK ACTIONS
    gr.Markdown(
        "## ⚡ Quick Research Actions"
    )

    with gr.Row():

        exp001_button = gr.Button(
            "🧪 Show EXP001"
        )

        exp002_button = gr.Button(
            "📊 Analyze EXP002"
        )

        exp003_button = gr.Button(
            "🧬 Show EXP003"
        )

        enzyme_button = gr.Button(
            "🔎 Enzyme Activity"
        )


    # SECOND QUICK ROW
    with gr.Row():

        growth_button = gr.Button(
            "🌱 Yeast Growth"
        )

        protein_button = gr.Button(
            "🧫 Protein Stability"
        )

        report_button = gr.Button(
            "📄 EXP002 Report"
        )


    # AGENT STATUS
    gr.Markdown(
        "## 🧠 Multi-Agent Architecture"
    )

    with gr.Row():

        gr.HTML("""
        <div class="card">
            <h3>🎯 Supervisor</h3>
            <p>
            Understands the request and
            selects the correct agent.
            </p>
        </div>
        """)

        gr.HTML("""
        <div class="card">
            <h3>📚 RAG Agent</h3>
            <p>
            Searches scientific knowledge
            using vector similarity.
            </p>
        </div>
        """)

        gr.HTML("""
        <div class="card">
            <h3>🧪 Experiment Agent</h3>
            <p>
            Retrieves structured experiment
            records from SQLite.
            </p>
        </div>
        """)

        gr.HTML("""
        <div class="card">
            <h3>📊 Analysis Agent</h3>
            <p>
            Performs statistics and
            machine-learning analysis.
            </p>
        </div>
        """)


    # DATABASE DASHBOARD
    gr.Markdown(
        "## 🗄️ Experiment Database"
    )

    database_table = gr.Dataframe(
        value=experiment_dashboard(),
        interactive=False,
        label="Synthetic Research Experiments"
    )

    refresh_button = gr.Button(
        "🔄 Refresh Experiments"
    )


    # SYSTEM INFORMATION
    gr.Markdown(
        "## 🔬 System Components"
    )

    gr.Markdown("""
### 🧠 Artificial Intelligence
The complete system acts as an intelligent research assistant.

### 🤖 Machine Learning
A Random Forest model demonstrates prediction of synthetic enzyme activity.

### 🧬 Deep Learning
`all-MiniLM-L6-v2` creates semantic embeddings using a pretrained transformer model.

### ✨ Generative AI / LLM
Qwen generates natural-language responses.

### 📚 RAG
Research documents are converted into chunks, embeddings are generated, and relevant chunks are retrieved from ChromaDB before generating an answer.

### 🕸️ LangGraph
The supervisor routes requests to specialized agents.

### 🔧 MCP
Research capabilities are exposed as standardized tools.

### 🗄️ SQLite
Stores structured researchers, experiments, results and research logs.
""")


    # --------------------------------------------------------
    # EVENTS
    # --------------------------------------------------------

    ask_button.click(
        chat_with_ai,
        inputs=[
            message_box,
            chatbot
        ],
        outputs=[
            chatbot,
            message_box
        ]
    )

    message_box.submit(
        chat_with_ai,
        inputs=[
            message_box,
            chatbot
        ],
        outputs=[
            chatbot,
            message_box
        ]
    )


    clear_button.click(
        lambda: [],
        outputs=chatbot
    )


    # QUICK QUESTIONS
    def quick_chat(
        question,
        history
    ):

        response = bioresearch_ai(
            question
        )

        history = history or []

        history.append({
            "role": "user",
            "content": question
        })

        history.append({
            "role": "assistant",
            "content": response
        })

        return history


    exp001_button.click(
        lambda history:
            quick_chat(
                "Show me EXP001",
                history
            ),
        inputs=chatbot,
        outputs=chatbot
    )

    exp002_button.click(
        lambda history:
            quick_chat(
                "Analyze EXP002",
                history
            ),
        inputs=chatbot,
        outputs=chatbot
    )

    exp003_button.click(
        lambda history:
            quick_chat(
                "Show me EXP003",
                history
            ),
        inputs=chatbot,
        outputs=chatbot
    )

    enzyme_button.click(
        lambda history:
            quick_chat(
                "What factors affect enzyme activity?",
                history
            ),
        inputs=chatbot,
        outputs=chatbot
    )

    growth_button.click(
        lambda history:
            quick_chat(
                "Explain factors affecting yeast growth",
                history
            ),
        inputs=chatbot,
        outputs=chatbot
    )

    protein_button.click(
        lambda history:
            quick_chat(
                "What factors affect protein stability?",
                history
            ),
        inputs=chatbot,
        outputs=chatbot
    )

    report_button.click(
        lambda history:
            quick_chat(
                "Generate a report for EXP002",
                history
            ),
        inputs=chatbot,
        outputs=chatbot
    )


    refresh_button.click(
        experiment_dashboard,
        outputs=database_table
    )


# ------------------------------------------------------------
# LAUNCH
# ------------------------------------------------------------

print("=" * 60)
print("🚀 LAUNCHING BIORESEARCH AI")
print("=" * 60)

demo.launch(
    share=True,
    debug=False
)

🚀 LAUNCHING BIORESEARCH AI
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://5e56ee4e55d1e56261.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
